# End-to-End ML Project: Customer Churn

This notebook demonstrates the full Snowflake ML workflow:
1. **Feature engineering** with Snowpark (pushdown transforms)
2. **Model training** with scikit-learn + Snowflake Experiment Tracking
3. **Model registration** in the Snowflake Model Registry
4. **Real-time inference** via a Model Service on Snowpark Container Services (SPCS)

Database/schema used: `ML_DEMO.PUBLIC`

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd
import numpy as np

session = get_active_session()
session.use_database("ML_DEMO")
session.use_schema("PUBLIC")

print(f"Database: {session.get_current_database()}, Schema: {session.get_current_schema()}")

In [ ]:
# Synthetic customer churn dataset (stand-in for a real source table)
np.random.seed(42)
n = 5000

tenure_months = np.random.randint(1, 72, n)
monthly_charges = np.round(np.random.uniform(20, 120, n), 2)
total_charges = np.round(monthly_charges * tenure_months * np.random.uniform(0.85, 1.0, n), 2)
support_calls = np.random.poisson(2, n)
contract_type = np.random.choice(["MONTH_TO_MONTH", "ONE_YEAR", "TWO_YEAR"], n, p=[0.55, 0.25, 0.20])
is_senior = np.random.choice([0, 1], n, p=[0.85, 0.15])

# Churn probability driven by tenure, support calls, and contract type
churn_logit = (
    -0.05 * tenure_months
    + 0.35 * support_calls
    + 0.01 * monthly_charges
    + np.where(contract_type == "MONTH_TO_MONTH", 1.2, 0)
    + np.where(contract_type == "ONE_YEAR", 0.2, 0)
    - 2.0
)
churn_prob = 1 / (1 + np.exp(-churn_logit))
churned = (np.random.uniform(0, 1, n) < churn_prob).astype(int)

raw_pdf = pd.DataFrame({
    "CUSTOMER_ID": np.arange(1, n + 1),
    "TENURE_MONTHS": tenure_months,
    "MONTHLY_CHARGES": monthly_charges,
    "TOTAL_CHARGES": total_charges,
    "SUPPORT_CALLS": support_calls,
    "CONTRACT_TYPE": contract_type,
    "IS_SENIOR": is_senior,
    "CHURNED": churned,
})

session.create_dataframe(raw_pdf).write.save_as_table("CUSTOMER_DATA", mode="overwrite")
print(f"Wrote {n} rows to ML_DEMO.PUBLIC.CUSTOMER_DATA")
print(f"Churn rate: {churned.mean():.2%}")

In [ ]:
from snowflake.snowpark.functions import col, when, iff, lit

raw = session.table("CUSTOMER_DATA")

# Feature engineering pushed down to Snowflake (no data pulled to pandas yet)
features = (
    raw
    .with_column("AVG_MONTHLY_SPEND", (col("TOTAL_CHARGES") / col("TENURE_MONTHS")).cast("float"))
    .with_column("HIGH_SUPPORT_USAGE", when(col("SUPPORT_CALLS") >= 3, lit(1)).otherwise(lit(0)))
    .with_column("IS_MONTH_TO_MONTH", when(col("CONTRACT_TYPE") == "MONTH_TO_MONTH", lit(1)).otherwise(lit(0)))
    .with_column("IS_ONE_YEAR", when(col("CONTRACT_TYPE") == "ONE_YEAR", lit(1)).otherwise(lit(0)))
    .with_column("IS_TWO_YEAR", when(col("CONTRACT_TYPE") == "TWO_YEAR", lit(1)).otherwise(lit(0)))
    .select(
        "CUSTOMER_ID", "TENURE_MONTHS", "MONTHLY_CHARGES", "TOTAL_CHARGES",
        "AVG_MONTHLY_SPEND", "SUPPORT_CALLS", "HIGH_SUPPORT_USAGE",
        "IS_SENIOR", "IS_MONTH_TO_MONTH", "IS_ONE_YEAR", "IS_TWO_YEAR", "CHURNED"
    )
)

features.write.save_as_table("CUSTOMER_FEATURES", mode="overwrite")
print(f"Feature table row count: {features.count()}")
features.limit(5).show()

In [ ]:
from sklearn.model_selection import train_test_split

FEATURE_COLS = [
    "TENURE_MONTHS", "MONTHLY_CHARGES", "TOTAL_CHARGES", "AVG_MONTHLY_SPEND",
    "SUPPORT_CALLS", "HIGH_SUPPORT_USAGE", "IS_SENIOR",
    "IS_MONTH_TO_MONTH", "IS_ONE_YEAR", "IS_TWO_YEAR",
]
TARGET_COL = "CHURNED"

# Small feature table (~5k rows) -> safe to pull fully to pandas for sklearn training
features_pdf = session.table("CUSTOMER_FEATURES").to_pandas()

X = features_pdf[FEATURE_COLS]
y = features_pdf[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train churn rate: {y_train.mean():.2%}, Test churn rate: {y_test.mean():.2%}")

In [ ]:
from snowflake.ml.experiment import ExperimentTracking
from snowflake.ml.model.model_signature import infer_signature
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

exp = ExperimentTracking(session=session, database_name="ML_DEMO", schema_name="PUBLIC")
exp.set_experiment("CHURN_EXPERIMENT")

params = {"n_estimators": 200, "max_depth": 8, "min_samples_leaf": 5, "random_state": 42}

with exp.start_run("rf_run_1"):
    exp.log_params(params)

    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "f1_score": f1_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_proba),
    }
    exp.log_metrics(metrics)

    sig = infer_signature(X_train, y_train)
    mv = exp.log_model(
        model,
        model_name="CHURN_MODEL",
        signatures={"predict": sig, "predict_proba": sig},
        conda_dependencies=["scikit-learn"],
        target_platforms=["WAREHOUSE", "SNOWPARK_CONTAINER_SERVICES"],
        comment="RandomForest churn classifier trained on synthetic CUSTOMER_FEATURES data",
    )

print("Metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}")
print(f"\nModel registered: {mv.model_name} version {mv.version_name}")

In [ ]:
import matplotlib.pyplot as plt

importances = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
importances.plot(kind="barh", ax=ax, color="#2E86AB")
ax.set_title("RandomForest Feature Importance — Customer Churn")
ax.set_xlabel("Importance")
plt.tight_layout()
plt.show()

In [ ]:
from snowflake.ml.registry import Registry

reg = Registry(session=session, database_name="ML_DEMO", schema_name="PUBLIC")
print(reg.show_models())
print(mv.show_functions())

In [ ]:
# NOTE: Service creation takes several minutes. This cell only submits the
# request; the next cell polls until the service reaches RUNNING.
# Using the CPU pool: this sklearn RandomForest model has no GPU requirement.
reg = Registry(session=session, database_name="ML_DEMO", schema_name="PUBLIC")
mv = reg.get_model("CHURN_MODEL").version(mv.version_name)

print("Creating service CHURN_INFERENCE_SERVICE...")
mv.create_service(
    service_name="CHURN_INFERENCE_SERVICE",
    service_compute_pool="SYSTEM_COMPUTE_POOL_CPU",
    ingress_enabled=True,
    max_instances=1,
    autocapture=True,
)
print("Service creation request submitted.")

In [ ]:
import time

MAX_WAIT_SECS = 1200  # 20 minutes
POLL_INTERVAL = 15
elapsed = 0

# If the service auto-suspended due to inactivity (auto_suspend_secs), resume it
# instead of treating SUSPENDED as a fatal error (auto_resume=true on this service).
rows = session.sql("DESCRIBE SERVICE ML_DEMO.PUBLIC.CHURN_INFERENCE_SERVICE").collect()
if rows[0]["status"] == "SUSPENDED":
    print("Service is SUSPENDED (auto-suspended from inactivity). Resuming...")
    session.sql("ALTER SERVICE ML_DEMO.PUBLIC.CHURN_INFERENCE_SERVICE RESUME").collect()

while elapsed < MAX_WAIT_SECS:
    rows = session.sql("DESCRIBE SERVICE ML_DEMO.PUBLIC.CHURN_INFERENCE_SERVICE").collect()
    status = rows[0]["status"]
    print(f"Service status: {status} (elapsed {elapsed}s)")

    if status == "RUNNING":
        break
    if status in ("FAILED", "DELETING"):
        raise RuntimeError(f"Service reached terminal status: {status}")

    time.sleep(POLL_INTERVAL)
    elapsed += POLL_INTERVAL
else:
    raise TimeoutError(f"Service did not reach RUNNING after {MAX_WAIT_SECS}s")

print("Service is RUNNING and ready for inference!")

In [ ]:
sample = X_test.head(5).reset_index(drop=True)

predictions = mv.run(sample, function_name="predict", service_name="CHURN_INFERENCE_SERVICE")
probabilities = mv.run(sample, function_name="predict_proba", service_name="CHURN_INFERENCE_SERVICE")

print("Sample input:")
print(sample)
print("\nPredictions:")
print(predictions)
print("\nProbabilities:")
print(probabilities)

In [ ]:
# The previous registration used a single `signatures` dict (inferred from y_train,
# 1 output column) for BOTH predict and predict_proba. predict_proba actually
# returns 2 columns (per-class probability), causing a signature mismatch at
# inference time. Re-log using sample_input_data so Snowflake infers the correct
# signature per method automatically (this also enables explainability).
reg = Registry(session=session, database_name="ML_DEMO", schema_name="PUBLIC")

mv = reg.log_model(
    model,
    model_name="CHURN_MODEL",
    sample_input_data=X_train,
    conda_dependencies=["scikit-learn"],
    target_platforms=["WAREHOUSE", "SNOWPARK_CONTAINER_SERVICES"],
    comment="RandomForest churn classifier (corrected signature inference)",
)

print(f"Model registered: {mv.model_name} version {mv.version_name}")
for func in mv.show_functions():
    print(f"\nFunction: {func['name']}")
    print(f"  Output features: {[f.name for f in func['signature'].outputs]}")

In [ ]:
session.sql("DROP SERVICE IF EXISTS ML_DEMO.PUBLIC.CHURN_INFERENCE_SERVICE").collect()

print("Creating service CHURN_INFERENCE_SERVICE with corrected model version...")
mv.create_service(
    service_name="CHURN_INFERENCE_SERVICE",
    service_compute_pool="SYSTEM_COMPUTE_POOL_CPU",
    ingress_enabled=True,
    max_instances=1,
    autocapture=True,
)
print("Service creation request submitted.")

In [ ]:
import time

MAX_WAIT_SECS = 1200
POLL_INTERVAL = 15
elapsed = 0

while elapsed < MAX_WAIT_SECS:
    rows = session.sql("DESCRIBE SERVICE ML_DEMO.PUBLIC.CHURN_INFERENCE_SERVICE").collect()
    status = rows[0]["status"]
    print(f"Service status: {status} (elapsed {elapsed}s)")

    if status == "RUNNING":
        break
    if status in ("FAILED", "DELETING", "SUSPENDED"):
        raise RuntimeError(f"Service reached terminal status: {status}")

    time.sleep(POLL_INTERVAL)
    elapsed += POLL_INTERVAL
else:
    raise TimeoutError(f"Service did not reach RUNNING after {MAX_WAIT_SECS}s")

print("Service is RUNNING and ready for inference!")

In [ ]:
sample = X_test.head(5).reset_index(drop=True)

predictions = mv.run(sample, function_name="predict", service_name="CHURN_INFERENCE_SERVICE")
probabilities = mv.run(sample, function_name="predict_proba", service_name="CHURN_INFERENCE_SERVICE")

print("Sample input:")
print(sample)
print("\nPredictions:")
print(predictions)
print("\nProbabilities:")
print(probabilities)

## Analyze Auto-Capture Inference Logs

Query `INFERENCE_TABLE()` for `CHURN_MODEL` to inspect captured request/response data from `CHURN_INFERENCE_SERVICE`.

In [ ]:
%%sql -r inference_logs_raw
SELECT
    TIMESTAMP,
    RECORD_ATTRIBUTES:"snow.model_serving.function.name"::STRING AS function_name,
    RECORD_ATTRIBUTES:"snow.model_serving.version"::STRING AS model_version,
    RECORD_ATTRIBUTES:"snow.model_serving.response.code"::STRING AS response_code,
    RECORD_ATTRIBUTES AS record_attributes
FROM TABLE(INFERENCE_TABLE('CHURN_MODEL'))
ORDER BY TIMESTAMP DESC

In [ ]:
import pandas as pd

logs_pdf = inference_logs_raw.copy() if isinstance(inference_logs_raw, pd.DataFrame) else inference_logs_raw.to_pandas()
print(f"Total captured inference events: {len(logs_pdf)}")

print("\nRequests by function:")
print(logs_pdf["FUNCTION_NAME"].value_counts())

print("\nRequests by model version:")
print(logs_pdf["MODEL_VERSION"].value_counts())

print("\nResponse code distribution:")
print(logs_pdf["RESPONSE_CODE"].value_counts())

In [ ]:
logs_pdf["HOUR"] = pd.to_datetime(logs_pdf["TIMESTAMP"]).dt.floor("h")
volume_by_hour = logs_pdf.groupby(["HOUR", "FUNCTION_NAME"]).size().unstack(fill_value=0)
print(volume_by_hour)

In [ ]:
import json

def extract_field(record_attrs, prefix):
    """Pull all request/response fields with a given prefix into a flat dict."""
    attrs = json.loads(record_attrs) if isinstance(record_attrs, str) else record_attrs
    return {k[len(prefix):]: v for k, v in attrs.items() if k.startswith(prefix)}

predict_rows = logs_pdf[logs_pdf["FUNCTION_NAME"] == "predict"]
if len(predict_rows) > 0:
    responses = predict_rows["RECORD_ATTRIBUTES"].apply(lambda r: extract_field(r, "snow.model_serving.response.data."))
    responses_df = pd.DataFrame(list(responses))
    print("predict() output value counts:")
    for col in responses_df.columns:
        print(f"\n{col}:")
        print(responses_df[col].value_counts())
else:
    print("No 'predict' function calls captured yet.")

proba_rows = logs_pdf[logs_pdf["FUNCTION_NAME"] == "predict_proba"]
if len(proba_rows) > 0:
    proba_responses = proba_rows["RECORD_ATTRIBUTES"].apply(lambda r: extract_field(r, "snow.model_serving.response.data."))
    proba_df = pd.DataFrame(list(proba_responses)).astype(float)
    print("\npredict_proba() output summary stats:")
    print(proba_df.describe())
else:
    print("No 'predict_proba' function calls captured yet.")

In [ ]:
sample_batch = X_test.head(30).reset_index(drop=True)
_ = mv.run(sample_batch, function_name="predict", service_name="CHURN_INFERENCE_SERVICE")
_ = mv.run(sample_batch, function_name="predict_proba", service_name="CHURN_INFERENCE_SERVICE")
print("Sent 30 predict + 30 predict_proba requests to CHURN_INFERENCE_SERVICE")

In [ ]:
import time
print("Waiting 90s for Auto-Capture events to land in the inference table...")
time.sleep(90)
print("Done waiting.")

In [ ]:
import requests

# NOTE: mv.run() calls the SQL service function wrapper, which does NOT
# get Auto-Captured (DISABLE_AUTOCAPTURE_FOR_SERVICE_FUNCTION=true in the
# service spec). Auto-Capture only logs raw REST calls to the service
# endpoint. From a Snowsight Notebook we can call the internal endpoint
# with no auth required.
services_df = mv.list_services()
print(services_df[["name", "status", "internal_endpoint", "autocapture_enabled"]])

internal_endpoint = services_df.loc[services_df["name"].str.contains("CHURN_INFERENCE_SERVICE"), "internal_endpoint"].iloc[0]
print(f"\nInternal endpoint: {internal_endpoint}")

sample_batch = X_test.head(20).reset_index(drop=True)
payload = {"data": [[i, *row] for i, row in enumerate(sample_batch.values.tolist())]}

for fn in ["predict", "predict_proba"]:
    url = f"{internal_endpoint}/{fn}"
    resp = requests.post(url, json=payload)
    print(f"{fn} -> status {resp.status_code}")

In [ ]:
resp = requests.post(f"{internal_endpoint}/predict-proba", json=payload)
print(f"predict-proba -> status {resp.status_code}")
if resp.status_code != 200:
    print(resp.text[:500])

In [ ]:
import time
print("Waiting 90s for Auto-Capture events to land in the inference table...")
time.sleep(90)
print("Done waiting.")

## Drift Detection: Model Version Monitor

Build a `SOURCE` table (logged predictions over time, from Auto-Capture) and a `BASELINE` table (predictions on training data), then create a `MODEL MONITOR` to track prediction-distribution drift for `CHURN_MODEL` against the training baseline.

In [ ]:
%%sql -r source_table_result
CREATE OR REPLACE TABLE ML_DEMO.PUBLIC.CHURN_PREDICTIONS_LOG AS
SELECT
    TIMESTAMP::TIMESTAMP_NTZ AS PRED_TIMESTAMP,
    RECORD_ATTRIBUTES:"snow.model_serving.request.data.TENURE_MONTHS"::NUMBER AS TENURE_MONTHS,
    RECORD_ATTRIBUTES:"snow.model_serving.request.data.MONTHLY_CHARGES"::FLOAT AS MONTHLY_CHARGES,
    RECORD_ATTRIBUTES:"snow.model_serving.request.data.TOTAL_CHARGES"::FLOAT AS TOTAL_CHARGES,
    RECORD_ATTRIBUTES:"snow.model_serving.request.data.AVG_MONTHLY_SPEND"::FLOAT AS AVG_MONTHLY_SPEND,
    RECORD_ATTRIBUTES:"snow.model_serving.request.data.SUPPORT_CALLS"::NUMBER AS SUPPORT_CALLS,
    RECORD_ATTRIBUTES:"snow.model_serving.request.data.HIGH_SUPPORT_USAGE"::NUMBER AS HIGH_SUPPORT_USAGE,
    RECORD_ATTRIBUTES:"snow.model_serving.request.data.IS_SENIOR"::NUMBER AS IS_SENIOR,
    RECORD_ATTRIBUTES:"snow.model_serving.request.data.IS_MONTH_TO_MONTH"::NUMBER AS IS_MONTH_TO_MONTH,
    RECORD_ATTRIBUTES:"snow.model_serving.request.data.IS_ONE_YEAR"::NUMBER AS IS_ONE_YEAR,
    RECORD_ATTRIBUTES:"snow.model_serving.request.data.IS_TWO_YEAR"::NUMBER AS IS_TWO_YEAR,
    RECORD_ATTRIBUTES:"snow.model_serving.response.data.output_feature_1"::FLOAT AS CHURN_SCORE
FROM TABLE(INFERENCE_TABLE('ML_DEMO.PUBLIC.CHURN_MODEL'))
WHERE RECORD_ATTRIBUTES:"snow.model_serving.function.name"::STRING = 'predict_proba'

In [ ]:
baseline_scores = model.predict_proba(X_train)[:, 1]

baseline_pdf = X_train.copy().reset_index(drop=True)
baseline_pdf["CHURN_SCORE"] = baseline_scores

session.create_dataframe(baseline_pdf).write.save_as_table("CHURN_BASELINE_PREDICTIONS", mode="overwrite")
print(f"Wrote {len(baseline_pdf)} baseline rows to ML_DEMO.PUBLIC.CHURN_BASELINE_PREDICTIONS")
print(baseline_pdf[["TENURE_MONTHS", "MONTHLY_CHARGES", "CHURN_SCORE"]].describe())

In [ ]:
%%sql -r create_monitor_result
CREATE MODEL MONITOR ML_DEMO.PUBLIC.CHURN_DRIFT_MONITOR WITH
    MODEL = ML_DEMO.PUBLIC.CHURN_MODEL
    VERSION = 'RUDE_SHEEP_2'
    FUNCTION = 'PREDICT_PROBA'
    SOURCE = ML_DEMO.PUBLIC.CHURN_PREDICTIONS_LOG
    WAREHOUSE = COMPUTE_WH
    REFRESH_INTERVAL = '1 hour'
    AGGREGATION_WINDOW = '7 days'
    TIMESTAMP_COLUMN = PRED_TIMESTAMP
    PREDICTION_SCORE_COLUMNS = ('CHURN_SCORE')
    BASELINE = ML_DEMO.PUBLIC.CHURN_BASELINE_PREDICTIONS

In [ ]:
%%sql -r drift_psi_result
SELECT *
FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
    'CHURN_DRIFT_MONITOR',
    'POPULATION_STABILITY_INDEX',
    'CHURN_SCORE',
    'DAY',
    DATEADD('day', -7, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ,
    CURRENT_TIMESTAMP()::TIMESTAMP_NTZ
))
ORDER BY 1

In [ ]:
%%sql -r drift_other_result
SELECT
    'JENSEN_SHANNON' AS metric, *
FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
    'CHURN_DRIFT_MONITOR', 'JENSEN_SHANNON', 'CHURN_SCORE', 'DAY',
    DATEADD('day', -7, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ, CURRENT_TIMESTAMP()::TIMESTAMP_NTZ
))
UNION ALL
SELECT
    'WASSERSTEIN' AS metric, *
FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
    'CHURN_DRIFT_MONITOR', 'WASSERSTEIN', 'CHURN_SCORE', 'DAY',
    DATEADD('day', -7, CURRENT_TIMESTAMP())::TIMESTAMP_NTZ, CURRENT_TIMESTAMP()::TIMESTAMP_NTZ
))

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

importances = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
importances.plot(kind="barh", ax=ax, color="#2E86AB")
ax.set_title("RandomForest Feature Importance — Customer Churn")
ax.set_xlabel("Importance")
plt.tight_layout()

import os
os.makedirs("images", exist_ok=True)
fig.savefig("images/feature_importance.png", dpi=150)
print("Saved to images/feature_importance.png")
print(os.path.abspath("images/feature_importance.png"))
plt.show()